## Run the full pipeline

Import required packages

In [ ]:
# for specifying the raw data location
from pathlib import Path

# for doing the processing
from rs_bidsify import processing, app_logging

Set input and output locations

In [ ]:
eeg_data_dir = Path("~/Documents/eeg_data").expanduser()

pavlov_raw = eeg_data_dir / "pavlov_data_single"

pavlov_out = eeg_data_dir / "pavlov_data_single_bids"

Optionally: Set-up logging to make the output more readable (and to save the complete log to a file)

In [ ]:
log_path = Path("~/Documents/eeg_data/logs").expanduser()

app_logging.setup_logging(log_path)

Run the pipeline

In [ ]:
pavlov_result = processing.process_dataset(pavlov_raw, pavlov_out, force_flag=True)

## Run individual steps

#### Setup
Import required modules and default config

In [ ]:
from mne_bids import BIDSPath, write_raw_bids

from rs_bidsify import discovery, enrichment, io
from rs_bidsify.validation.dataset import EEGDatasetCrawler
from rs_bidsify.validation.subject import SubjectMetadata

In [ ]:
default_config = processing.get_default_config()
config = default_config.copy()

#### Validate

Validate the input data:
- Dataset specification (JSON)
- Dataset spreadsheet
- Directory structure

In [ ]:

# Dataset specification
dataset_spec = discovery.find_description_spec(pavlov_raw, extension=config["metadata_ext"])

# Dataset spreadsheet
participant_data, phenotype_data = discovery.find_dataset_spreadsheets(
        pavlov_raw, sheet_info=config["sheet_info"], extension=config["spreadsheet_ext"]
    )

expected_participants = participant_data["dataset"].index.to_list()

# Directory structure 
crawler = EEGDatasetCrawler(
    root_path=pavlov_raw,
    expected_participants=expected_participants,
    **dataset_spec.crawler_info,
)

#### Convert

Let's just process the first recording

In [ ]:
recording = crawler.found_recordings[0]

Set subject level information

In [ ]:
subject_bids_path = BIDSPath(
    subject=recording.subject,
    task=recording.task,
    root=pavlov_out,
)

subject_info = SubjectMetadata.from_dataframe(
    recording,
    participant_data["dataset"],
    mapping=config["demographic_mappings"],
)

Load the recording, update the MNE object, and save to a BIDS format

In [ ]:
eeg_data = io.read_eeg_recording(recording.path)

enrichment.set_subject_info(eeg_data, subject_info)

enrichment.enrich_mne_object(eeg_data, dataset_spec)

rec_bids_path = write_raw_bids(
    eeg_data,
    subject_bids_path,
    overwrite=True,
    allow_preload=True,
    format=config["output_EEG_format"].upper()
)

#### Enrich

Add more metadata to the existing files

In [ ]:
# Update sidecar
enrichment.enrich_eeg_sidecar(rec_bids_path, dataset_spec, config["include_extras"])

# Update channels (where required)
enrichment.enrich_channels_tsv_with_aux(rec_bids_path, dataset_spec.acquisition_spec.aux_channels)

## Use a custom config

Import packages for loading custom config 

In [ ]:
import yaml
from rs_bidsify.config_loader import deep_merge

Moving to different dataset, set new input and output paths

In [ ]:
eimer_raw = eeg_data_dir / "eimer_raw"
eimer_out = eeg_data_dir / "eimer_bids"

In [ ]:


config_path = discovery.find_file(eimer_raw, "yaml")

with open(config_path) as config:
    custom_config = yaml.safe_load(config)

config = deep_merge(default_config, custom_config)

In [ ]:
eimer_result = processing.process_dataset(
    eimer_raw, 
    eimer_out, 
    config_override=config, 
    force_flag=True)